In [6]:
import pandas as pd
import numpy as np

# Load the original dataset
df = pd.read_csv(
    r"I:\Muzzamil Data Analyst\OBSIS Intership Project\OIBSIP DataAnalytics-L1-CleaningData\train.csv"
)


In [8]:
# Keep a copy of the original data
df_original = df.copy()

# Inspect the dataset
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

display(df.head())

Dataset shape: (891, 12)

Column names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Missing values per column:


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


Duplicate rows: 0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().mean() * 100
    ).round(2),
    "Unique Values": df.nunique(),
    "Duplicate Count": [df.duplicated().sum()] * len(df.columns)
})

display(quality_report)

,Data Type,Missing Values,Missing Percentage,Unique Values,Duplicate Count
PassengerId,int64,0,0.00,891,0
Survived,int64,0,0.00,2,0
Pclass,int64,0,0.00,3,0
Name,object,0,0.00,891,0
Sex,object,0,0.00,2,0
Age,float64,177,19.87,88,0
SibSp,int64,0,0.00,7,0
Parch,int64,0,0.00,7,0
Ticket,object,0,0.00,681,0
Fare,float64,0,0.00,248,0


## Missing Data Handling — Decisions and Justifications

1. **Age:** 177 values (19.87%) are missing. These values will be imputed using the median because age is numerical and the median is less sensitive to extreme values.
2. **Cabin:** 687 values (77.10%) are missing. The original missingness will be preserved in a new `Cabin_Missing` indicator column, and missing cabin entries will be represented as `"Unknown"` rather than inventing cabin numbers.
3. **Embarked:** Two values (0.22%) are missing. These will be filled using the mode, which is the most frequent embarkation category.
4. **Duplicates:** No exact duplicate rows were found, so no rows will be removed for duplication.
5. **Other columns:** No missing values were found, so no missing-value treatment is required for those columns.

All decisions will be documented so that the cleaning process remains transparent and reproducible.

In [13]:
# Create a working copy
df_clean = df.copy()

# 1. Create an indicator for missing cabin information
df_clean["Cabin_Missing"] = df_clean["Cabin"].isna().astype(int)

# 2. Fill missing Age values with the median
age_median = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(age_median)

# 3. Fill missing Cabin values with "Unknown"
df_clean["Cabin"] = df_clean["Cabin"].fillna("Unknown")

# 4. Fill missing Embarked values with the mode
embarked_mode = df_clean["Embarked"].mode()[0]
df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode)

# Verify missing values
print("Missing values after imputation:")
display(df_clean.isnull().sum())

print("\nAge median used:", age_median)
print("Embarked mode used:", embarked_mode)
print("Rows after imputation:", len(df_clean))

Missing values after imputation:


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin            0
Embarked         0
Cabin_Missing    0
dtype: int64


Age median used: 28.0
Embarked mode used: S
Rows after imputation: 891


In [18]:
print("Data types after imputation:")
print(df_clean.dtypes)

print("\nSex values:")
print(df_clean["Sex"].value_counts(dropna=False))

print("\nEmbarked values:")
print(df_clean["Embarked"].value_counts(dropna=False))

print("\nPclass values:")
print(df_clean["Pclass"].value_counts(dropna=False))

print("\nNumeric ranges:")
display(df_clean[["Age", "SibSp", "Parch", "Fare"]].describe())

Data types after imputation:
PassengerId        int64
Survived           int64
Pclass             int64
Name              object
Sex               object
Age              float64
SibSp              int64
Parch              int64
Ticket            object
Fare             float64
Cabin             object
Embarked          object
Cabin_Missing      int32
dtype: object

Sex values:
Sex
male      577
female    314
Name: count, dtype: int64

Embarked values:
Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

Pclass values:
Pclass
3    491
1    216
2    184
Name: count, dtype: int64

Numeric ranges:


,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,891.000000
mean,29.361582,0.523008,0.381594,32.204208
std,13.019697,1.102743,0.806057,49.693429
min,0.420000,0.000000,0.000000,0.000000
25%,22.000000,0.000000,0.000000,7.910400
50%,28.000000,0.000000,0.000000,14.454200
75%,35.000000,1.000000,0.000000,31.000000
max,80.000000,8.000000,6.000000,512.329200


### Standardisation and Data Type Review

1. The `Sex` column contains two consistent categories: `male` and `female`.
2. The `Embarked` column contains three categories: `S`, `C`, and `Q`.
3. The `Pclass` column contains passenger classes 1, 2, and 3.
4. The `Age` column is numerical, and `PassengerId` is an integer identifier.
5. `Cabin_Missing` records whether the original Cabin value was missing.
6. The `Fare` column contains a minimum value of 0 and a maximum of 512.33. These values will be investigated before deciding whether any observations require special treatment.

In [21]:
# Standardise text formatting
df_clean["Sex"] = df_clean["Sex"].str.strip().str.lower()
df_clean["Embarked"] = df_clean["Embarked"].str.strip().str.upper()

# Correct data types
df_clean["PassengerId"] = df_clean["PassengerId"].astype("int64")
df_clean["Survived"] = df_clean["Survived"].astype("int64")
df_clean["Pclass"] = df_clean["Pclass"].astype("int64")

df_clean["Age"] = pd.to_numeric(df_clean["Age"], errors="coerce")
df_clean["Fare"] = pd.to_numeric(df_clean["Fare"], errors="coerce")

# IDs and codes should be strings, not measurements
df_clean["Ticket"] = df_clean["Ticket"].astype("string")
df_clean["Cabin"] = df_clean["Cabin"].astype("string")
df_clean["Sex"] = df_clean["Sex"].replace({
    "male": "Male",
    "female": "Female"
})

print(df_clean.dtypes)

PassengerId               int64
Survived                  int64
Pclass                    int64
Name                     object
Sex                      object
Age                     float64
SibSp                     int64
Parch                     int64
Ticket           string[python]
Fare                    float64
Cabin            string[python]
Embarked                 object
Cabin_Missing             int32
dtype: object


In [23]:
numeric_cols = ["Age", "SibSp", "Parch", "Fare"]

outlier_results = []

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    )

    outlier_results.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outliers.sum())
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report.round(2))

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Age,22.00,35.0,13.00,2.50,54.50,66
1,SibSp,0.00,1.0,1.00,-1.50,2.50,46
2,Parch,0.00,0.0,0.00,0.00,0.00,213
3,Fare,7.91,31.0,23.09,-26.72,65.63,116


### Outlier Detection — IQR Method

The Interquartile Range (IQR) method was applied to Age, SibSp, Parch, and Fare.

| Column | Outliers detected | Treatment |
|---|---:|---|
| Age | 66 | Retain plausible ages |
| SibSp | 46 | Retain plausible family counts |
| Parch | 213 | Retain; IQR equals zero because Q1 and Q3 are zero |
| Fare | 116 | Retain plausible fares and investigate unusual values |

**Decision:** No observations were removed or capped solely because the IQR method flagged them. The dataset represents historical passenger records, and unusual values may be valid. The IQR method identifies statistical outliers, not necessarily data errors.

In [26]:
# Inspect the 10 highest fares
display(
    df_clean.nlargest(10, "Fare")[
        ["PassengerId", "Pclass", "Sex", "Age", "Fare", "Embarked"]
    ]
)

# Inspect unusually large family counts
display(
    df_clean.nlargest(10, "SibSp")[
        ["PassengerId", "Name", "SibSp", "Parch"]
    ]
)

# Verify age range
print("Minimum age:", df_clean["Age"].min())
print("Maximum age:", df_clean["Age"].max())

,PassengerId,Pclass,Sex,Age,Fare,Embarked
258,259,1,Female,35.0,512.3292,C
679,680,1,Male,36.0,512.3292,C
737,738,1,Male,35.0,512.3292,C
27,28,1,Male,19.0,263.0000,S
88,89,1,Female,23.0,263.0000,S
341,342,1,Female,24.0,263.0000,S
438,439,1,Male,64.0,263.0000,S
311,312,1,Female,18.0,262.3750,C
742,743,1,Female,21.0,262.3750,C
118,119,1,Male,24.0,247.5208,C


,PassengerId,Name,SibSp,Parch
159,160,"Sage, Master. Thomas Henry",8,2
180,181,"Sage, Miss. Constance Gladys",8,2
201,202,"Sage, Mr. Frederick",8,2
324,325,"Sage, Mr. George John Jr",8,2
792,793,"Sage, Miss. Stella Anna",8,2
846,847,"Sage, Mr. Douglas Bullen",8,2
863,864,"Sage, Miss. Dorothy Edith ""Dolly""",8,2
59,60,"Goodwin, Master. William Frederick",5,2
71,72,"Goodwin, Miss. Lillian Amy",5,2
386,387,"Goodwin, Master. Sidney Leonard",5,2


Minimum age: 0.42
Maximum age: 80.0


In [28]:
before_after = pd.DataFrame({
    "Metric": [
        "Row count",
        "Column count",
        "Duplicate rows",
        "Missing Age",
        "Missing Cabin",
        "Missing Embarked",
        "Total missing cells"
    ],
    "Before": [
        len(df_original),
        len(df_original.columns),
        df_original.duplicated().sum(),
        df_original["Age"].isna().sum(),
        df_original["Cabin"].isna().sum(),
        df_original["Embarked"].isna().sum(),
        df_original.isna().sum().sum()
    ],
    "After": [
        len(df_clean),
        len(df_clean.columns),
        df_clean.duplicated().sum(),
        df_clean["Age"].isna().sum(),
        df_clean["Cabin"].isna().sum(),
        df_clean["Embarked"].isna().sum(),
        df_clean.isna().sum().sum()
    ]
})

display(before_after)

,Metric,Before,After
0,Row count,891,891
1,Column count,12,13
2,Duplicate rows,0,0
3,Missing Age,177,0
4,Missing Cabin,687,0
5,Missing Embarked,2,0
6,Total missing cells,866,0


In [30]:
import os

output_path = (
    r"I:\Muzzamil Data Analyst\OBSIS Intership Project"
    r"\DataAnalytics-L1-CleaningData"
)

# Create output folder if it doesn't exist
os.makedirs(output_path, exist_ok=True)

# Save cleaned dataset separately
csv_path = os.path.join(output_path, "Titanic_Cleaned.csv")
df_clean.to_csv(csv_path, index=False)

print("Cleaned dataset saved successfully:")
print(csv_path)
print("Final shape:", df_clean.shape)

Cleaned dataset saved successfully:
I:\Muzzamil Data Analyst\OBSIS Intership Project\DataAnalytics-L1-CleaningData\Titanic_Cleaned.csv
Final shape: (891, 13)


In [32]:
df_check = pd.read_csv(csv_path)

print("Saved file shape:", df_check.shape)
print("Missing values:", df_check.isnull().sum().sum())
print("Duplicate rows:", df_check.duplicated().sum())

display(df_check.head())

Saved file shape: (891, 13)
Missing values: 0
Duplicate rows: 0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Cabin_Missing
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,Unknown,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.2833,C85,C,0
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,S,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,C123,S,0
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.0500,Unknown,S,1


# Conclusion — Titanic Data Cleaning

This project demonstrated a systematic data cleaning workflow using Python, Pandas, and NumPy.

The original Titanic dataset was inspected for missing values, duplicate records, data types, inconsistent formatting, and potential numerical outliers.

Missing Age values were filled using the median, missing Embarked values using the mode, and missing Cabin values with `"Unknown"`. A `Cabin_Missing` indicator was created to preserve information about missing cabin records. Data types and categorical formatting were standardized.

The IQR method was used to identify potential outliers in Age, SibSp, Parch, and Fare. These observations were retained unless there was evidence that a value was invalid, because statistical outliers may represent legitimate passenger records.

Finally, a before-versus-after quality summary was prepared and the cleaned dataset was saved as a separate CSV file.

The resulting dataset is better organized for further analysis while retaining potentially meaningful passenger information.